In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense

# 1. Load and preprocess data
df = pd.read_csv("Datasets/AutoEncoder/Energy Consumption/energy consumption.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True)
df = df.sort_values("Timestamp").ffill().fillna(0)

features = ["Electricity_Consumed", "Temperature", "Humidity", "Wind_Speed", "Avg_Past_Consumption"]

scaler = MinMaxScaler()
data = scaler.fit_transform(df[features])

# 2. Create time windows
window = 10
X = np.array([data[i:i+window] for i in range(len(data)-window+1)])

# Train only on completely normal windows
labels = df["Anomaly_Label"].values

normal = np.array([
    np.all(labels[i:i+window] == 0)
    for i in range(len(X))
])

X_train = X[normal]

# 3. Build LSTM autoencoder
inp = Input(shape=(window, len(features)))
x = LSTM(32, activation="relu")(inp)
x = RepeatVector(window)(x)
x = LSTM(32, activation="relu", return_sequences=True)(x)
out = TimeDistributed(Dense(len(features)))(x)

model = Model(inp, out)
model.compile(optimizer="adam", loss="mse")

# 4. Train the model
model.fit(X_train, X_train, epochs=20, batch_size=32, validation_split=0.1)

# 5. Calculate reconstruction error and detect anomalies
reconstructed = model.predict(X, verbose=0)
error = np.mean((X - reconstructed) ** 2, axis=(1, 2))

threshold = np.percentile(error[normal], 95)
anomalies = error > threshold

# 6. Display results
df["Detected_Anomaly"] = False
df.loc[window-1:, "Detected_Anomaly"] = anomalies

print("Threshold:", threshold)
print(df[["Timestamp", "Electricity_Consumed", "Anomaly_Label", "Detected_Anomaly"]].tail(10))

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.1482 - val_loss: 0.0233
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0206 - val_loss: 0.0168
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0168 - val_loss: 0.0160
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0162 - val_loss: 0.0155
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0156 - val_loss: 0.0154
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0154 - val_loss: 0.0151
Epoch 7/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0151 - val_loss: 0.0150
Epoch 8/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0148 - val_loss: 0.0147
Epoch 9/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0146 - val_loss: 0.0146
Epoch 10/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0144 - val_loss: 0.0143
Epoch 11/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0142 - val_loss: 0.0144
Epoch 12/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/ste

In [6]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Input, Dense, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras.models import Model

df = pd.read_csv("Datasets/AutoEncoder/Energy Consumption/energy consumption.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True)
df = df.sort_values("Timestamp").ffill().fillna(0)

features = ["Electricity_Consumed", "Temperature", "Humidity", "Wind_Speed", "Avg_Past_Consumption"]

scaler = MinMaxScaler()
data = scaler.fit_transform(df[features])

window = 10
X = np.array([data[i : i + window] for i in range(len(data) - window + 1)])

labels = df["Anomaly_Label"].values

nor = np.array([np.all(labels[i : i + window] == 0) for i in range(len(X))])

X_train = X[nor]

inp = Input(shape=(window, len(features)))
en = LSTM(32, activation='relu')(inp)
en = RepeatVector(window)(en)
en = LSTM(32, activation='relu', return_sequences=True)(en)
out = TimeDistributed(Dense(len(features)))(en)

model = Model(inp, out)

model.compile(
    optimizer='adam',
    loss='mse'
)

model.fit(X_train, X_train, epochs=20, batch_size=32, validation_split=0.1)

reconstruct = model.predict(X, verbose=0)
error = np.mean((X - reconstructed) ** 2, axis=(1, 2))

threshold = np.percentile(error[nor], 95)
anomalies = error > threshold

df["Detected_Anomaly"] = False
df.loc[window - 1 :, "Detected_Anomaly"] = anomalies

print("Threshold: ", threshold)
print(df[["Electricity_Consumed", "Temperature", "Humidity", "Wind_Speed", "Avg_Past_Consumption", "Anomaly_Label", "Detected_Anomaly"]].tail(10))

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - loss: 0.1511 - val_loss: 0.0247
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0226 - val_loss: 0.0175
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0172 - val_loss: 0.0159
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0160 - val_loss: 0.0156
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0157 - val_loss: 0.0154
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0155 - val_loss: 0.0152
Epoch 7/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0152 - val_loss: 0.0156
Epoch 8/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0151 - val_loss: 0.0148
Epoch 9/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0148 - val_loss: 0.0147
Epoch 10/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0147 - val_loss: 0.0150
Epoch 11/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0145 - val_loss: 0.0145
Epoch 12/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/ste